In [11]:
from pathlib import Path
from typing import Literal
import torch

from transformers import AutoModelForTokenClassification, AutoTokenizer

# Load test dataset with the same random seed for data splitting
from spesia_research.datasets import ClinicalRecordsDataset, DataCollatorForMultiLabelTokenClassification

mutually_exclusive_classes = [
    ["HER2_NEGATIVO", "HER2_POSITIVO"],
    ['RP_NEGATIVO', 'RP_POSITIVO'],
    ['RE_NEGATIVO', 'RE_POSITIVO'],
]

mutually_exclusive_classes_unpacked = []
for g in mutually_exclusive_classes:
    mutually_exclusive_classes_unpacked.extend(g)

random_seed = 3
batch_size = 15
max_length = 1024
dataset_path = Path('datasets/breast_cancer_dataset')
model_id = r"experiments\random_seed_3\bce_with_grouped_softmax_random_seed_3\best_model"
device = torch.device('cuda') if torch.cuda.is_available() else torch.device("cpu")

model = AutoModelForTokenClassification.from_pretrained(model_id).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

test_dataset = ClinicalRecordsDataset(dataset_path, split='test', random_seed=random_seed, tokenizer=tokenizer)

data_collator = DataCollatorForMultiLabelTokenClassification(tokenizer.pad_token_id, max_length=max_length, num_labels=test_dataset.num_labels, device=device)

# preds for non-exclusive labels
best_thresholds = [
    v["threshold"] for k, v in model.config.thresholds.items() if k not in mutually_exclusive_classes_unpacked
]

probability_decoding_strategy: Literal["sigmoid", "grouped_softmax"] = "grouped_softmax"

Loading all Records: 100%|██████████| 1/1 [00:02<00:00,  2.04s/it]


In [ ]:
from transformers import EvalPrediction

with torch.no_grad():
    agg_labels = None
    agg_logits = None
    for idx in range(0, len(test_dataset), batch_size):
        batch = test_dataset[idx:idx+batch_size]
        batch = data_collator(batch)
        labels = batch.pop("labels")
        output = model(**batch)
        logits = output.logits

        if agg_labels is None and agg_logits is None:
            agg_labels = labels
            agg_logits = logits
        else:
            agg_labels = torch.cat((agg_labels, labels), dim=0)
            agg_logits = torch.cat((agg_logits, logits), dim=0)

In [13]:
# Token level metrics

from spesia_research.metrics import CustomMetricsForGroupedSoftmax, compute_metrics_with_per_label_thresholds


eval_preds = EvalPrediction(predictions=agg_logits.detach().cpu().numpy(), label_ids=agg_labels.detach().cpu().numpy())

if probability_decoding_strategy == "grouped_softmax":
    custom_metrics = CustomMetricsForGroupedSoftmax(mutually_exclusive_classes, model.config.label2id)
    metrics = custom_metrics(eval_preds, include_per_label_thresholds=True)
    print(metrics)

elif probability_decoding_strategy == "sigmoid":
    custom_metrics = compute_metrics_with_per_label_thresholds
    metrics = custom_metrics(eval_preds, include_per_label_thresholds=True)
    print(metrics)

{'best_thresholds': tensor([0.8600, 0.9500, 0.5300, 0.3000, 0.6800, 0.9000, 0.9300, 0.5000, 0.4100,
        0.4900, 0.4500, 0.3700]), 'best_thresholds_mean': 0.6141666769981384, 'macro_precision': 0.713766264174708, 'macro_recall': 0.7737497058350672, 'macro_f1': 0.7395698423682696, 'micro_precision': 0.7780840039686915, 'micro_recall': 0.8552041681812674, 'micro_f1': 0.8148233664280766, 'macro_pr_auc': 0.7490826459717081, 'micro_pr_auc': 0.7862338633075615, 'precision_BRCA_NEGATIVO': 0.5869565217391305, 'recall_BRCA_NEGATIVO': 0.6026785714285714, 'f1_BRCA_NEGATIVO': 0.5947136563876652, 'precision_BRCA_POSITIVO': 0.575, 'recall_BRCA_POSITIVO': 0.5542168674698795, 'f1_BRCA_POSITIVO': 0.5644171779141104, 'precision_CIRURGIA': 0.8155067725361981, 'recall_CIRURGIA': 0.8686567164179104, 'f1_CIRURGIA': 0.8412430739580824, 'precision_HER2_NEGATIVO': 0.875, 'recall_HER2_NEGATIVO': 0.7509578544061303, 'f1_HER2_NEGATIVO': 0.8082474226804124, 'precision_HER2_POSITIVO': 0.6717892425905598, 'recall

In [14]:
# Span level metrics

from spesia_research.metrics import BCESpanLevelMetrics, GroupedSoftmaxSpanLevelMetrics

eval_preds = EvalPrediction(predictions=agg_logits, label_ids=agg_labels)

if probability_decoding_strategy == "grouped_softmax":
    custom_metrics = GroupedSoftmaxSpanLevelMetrics(mutually_exclusive_classes, model.config.id2label, mode="strict", prediction_thresholds=best_thresholds)
    metrics = custom_metrics(eval_preds, include_per_label_thresholds=True)
    print(metrics)

elif probability_decoding_strategy == "sigmoid":
    custom_metrics = BCESpanLevelMetrics(best_thresholds, model.config.id2label, mode="strict")
    metrics = custom_metrics(eval_preds, include_per_label_thresholds=True)
    print(metrics)

c:\Users\almei\Documents\GitHub\research_mecla_objective_paper\spesia_research\metrics.py:615: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  logits = torch.tensor(eval_pred.predictions)
c:\Users\almei\Documents\GitHub\research_mecla_objective_paper\spesia_research\metrics.py:616: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(eval_pred.label_ids)


{'micro_precision': 0.567922874671341, 'micro_recall': 0.6941617568291376, 'micro_f1': 0.6247288503253796, 'macro_precision': 0.4833119682113869, 'macro_recall': 0.6460725714044097, 'macro_f1': 0.5406255776613951, 'metrics_per_entity': {'BRCA_NEGATIVO': {'tp': 5, 'fp': 47, 'fn': 15, 'precision': 0.09615384615384616, 'recall': 0.25, 'f1': 0.1388888888888889}, 'BRCA_POSITIVO': {'tp': 6, 'fp': 27, 'fn': 6, 'precision': 0.18181818181818182, 'recall': 0.5, 'f1': 0.26666666666666666}, 'CIRURGIA': {'tp': 289, 'fp': 202, 'fn': 185, 'precision': 0.5885947046843177, 'recall': 0.609704641350211, 'f1': 0.5989637305699482}, 'HER2_NEGATIVO': {'tp': 146, 'fp': 45, 'fn': 46, 'precision': 0.7643979057591623, 'recall': 0.7604166666666666, 'f1': 0.7624020887728459}, 'HER2_POSITIVO': {'tp': 93, 'fp': 82, 'fn': 21, 'precision': 0.5314285714285715, 'recall': 0.8157894736842105, 'f1': 0.6435986159169551}, 'POS_MENOPAUSA': {'tp': 36, 'fp': 79, 'fn': 18, 'precision': 0.3130434782608696, 'recall': 0.66666666666

## BCE + MECLA + Sigmoid decoding

Span level analysis
---

### Global Metrics

| Metric            | Value     |
|------------------|----------:|
| micro_precision  | 0.5457    |
| micro_recall     | 0.7097    |
| micro_f1         | 0.6170    |
| macro_precision  | 0.4563    |
| macro_recall     | 0.6194    |
| macro_f1         | 0.5170    |

### Per-Entity Metrics

| Entity                     | TP  | FP  | FN  | Precision | Recall | F1     |
|----------------------------|----:|----:|----:|----------:|-------:|-------:|
| BRCA_NEGATIVO              |   3 |  67 |  17 | 0.0429    | 0.1500 | 0.0667 |
| BRCA_POSITIVO              |   4 |  32 |   8 | 0.1111    | 0.3333 | 0.1667 |
| CIRURGIA                   | 356 | 191 | 118 | 0.6508    | 0.7511 | 0.6974 |
| HER2_NEGATIVO              | 145 | 125 |  47 | 0.5370    | 0.7552 | 0.6277 |
| HER2_POSITIVO              |  84 |  69 |  30 | 0.5490    | 0.7368 | 0.6292 |
| POS_MENOPAUSA              |  30 |  48 |  24 | 0.3846    | 0.5556 | 0.4545 |
| PRE_MENOPAUSA              |   6 |  80 |  14 | 0.0698    | 0.3000 | 0.1132 |
| RE_NEGATIVO                |  51 |  35 |  11 | 0.5930    | 0.8226 | 0.6892 |
| RE_POSITIVO                | 189 |  82 |  40 | 0.6974    | 0.8253 | 0.7560 |
| RP_NEGATIVO                |  76 |  51 |  22 | 0.5984    | 0.7755 | 0.6756 |
| RP_POSITIVO                | 162 |  39 |  19 | 0.8060    | 0.8950 | 0.8482 |
| TIPO_HISTOPATOLOGICO       | 219 | 284 | 192 | 0.4354    | 0.5328 | 0.4792 |

## BCE + MECLA + Grouped Softmax probability decoding

Token level analysis
---

### Global Metrics

| Metric                 | Value     |
|------------------------|----------:|
| micro_precision        | 0.7924    |
| micro_recall           | 0.8248    |
| micro_f1               | 0.8083    |
| macro_precision        | 0.7184    |
| macro_recall           | 0.7314    |
| macro_f1               | 0.7197    |
| macro_pr_auc           | 0.7165    |
| micro_pr_auc           | 0.7271    |
| best_thresholds_mean   | 0.6200    |

### Per-Entity Metrics

| Entity                     | Precision | Recall | F1     |
|----------------------------|----------:|-------:|-------:|
| BRCA_NEGATIVO              | 0.6733    | 0.4509 | 0.5401 |
| BRCA_POSITIVO              | 0.4434    | 0.5663 | 0.4974 |
| CIRURGIA                   | 0.8380    | 0.8547 | 0.8463 |
| HER2_NEGATIVO              | 0.8435    | 0.7573 | 0.7981 |
| HER2_POSITIVO              | 0.6882    | 0.8997 | 0.7799 |
| POS_MENOPAUSA              | 0.6947    | 0.6839 | 0.6893 |
| PRE_MENOPAUSA              | 0.2321    | 0.3333 | 0.2737 |
| RE_NEGATIVO                | 0.7798    | 0.8374 | 0.8076 |
| RE_POSITIVO                | 0.9064    | 0.9281 | 0.9171 |
| RP_NEGATIVO                | 0.8295    | 0.7254 | 0.7740 |
| RP_POSITIVO                | 0.9365    | 0.8939 | 0.9147 |
| TIPO_HISTOPATOLOGICO       | 0.7557    | 0.8463 | 0.7985 |


Span level analysis
---

### Global Metrics

| Metric            | Value     |
|------------------|----------:|
| micro_precision  | 0.5565    |
| micro_recall     | 0.6963    |
| micro_f1         | 0.6186    |
| macro_precision  | 0.4683    |
| macro_recall     | 0.5989    |
| macro_f1         | 0.5148    |

### Per-Entity Metrics

| Entity                     | TP  | FP  | FN  | Precision | Recall | F1     |
|----------------------------|----:|----:|----:|----------:|-------:|-------:|
| BRCA_NEGATIVO              |   3 |  67 |  17 | 0.0429    | 0.1500 | 0.0667 |
| BRCA_POSITIVO              |   4 |  32 |   8 | 0.1111    | 0.3333 | 0.1667 |
| CIRURGIA                   | 356 | 191 | 118 | 0.6508    | 0.7511 | 0.6974 |
| HER2_NEGATIVO              | 141 |  61 |  51 | 0.6980    | 0.7344 | 0.7157 |
| HER2_POSITIVO              |  87 |  96 |  27 | 0.4754    | 0.7632 | 0.5859 |
| POS_MENOPAUSA              |  30 |  48 |  24 | 0.3846    | 0.5556 | 0.4545 |
| PRE_MENOPAUSA              |   6 |  80 |  14 | 0.0698    | 0.3000 | 0.1132 |
| RE_NEGATIVO                |  43 |  32 |  19 | 0.5733    | 0.6935 | 0.6277 |
| RE_POSITIVO                | 191 |  69 |  38 | 0.7346    | 0.8341 | 0.7812 |
| RP_NEGATIVO                |  69 |  38 |  29 | 0.6449    | 0.7041 | 0.6732 |
| RP_POSITIVO                | 151 |  38 |  30 | 0.7989    | 0.8343 | 0.8162 |
| TIPO_HISTOPATOLOGICO       | 219 | 284 | 192 | 0.4354    | 0.5328 | 0.4792 |

## BCE + Pairwise MECLA + Grouped Softmax probability decoding

### Global Metrics

| Metric                 | Value     |
|------------------------|----------:|
| micro_precision        | 0.7618    |
| micro_recall           | 0.8376    |
| micro_f1               | 0.7979    |
| macro_precision        | 0.6767    |
| macro_recall           | 0.7712    |
| macro_f1               | 0.7162    |
| macro_pr_auc           | 0.7227    |
| micro_pr_auc           | 0.7885    |
| best_thresholds_mean   | 0.5800    |

### Per-Entity Metrics

| Entity                     | Precision | Recall | F1     |
|----------------------------|----------:|-------:|-------:|
| BRCA_NEGATIVO              | 0.3521    | 0.6429 | 0.4550 |
| BRCA_POSITIVO              | 0.3864    | 0.6145 | 0.4744 |
| CIRURGIA                   | 0.8247    | 0.8542 | 0.8392 |
| HER2_NEGATIVO              | 0.7655    | 0.8046 | 0.7846 |
| HER2_POSITIVO              | 0.7028    | 0.8119 | 0.7535 |
| POS_MENOPAUSA              | 0.5714    | 0.7047 | 0.6311 |
| PRE_MENOPAUSA              | 0.3630    | 0.4530 | 0.4030 |
| RE_NEGATIVO                | 0.7342    | 0.8571 | 0.7909 |
| RE_POSITIVO                | 0.9280    | 0.9260 | 0.9270 |
| RP_NEGATIVO                | 0.8089    | 0.8034 | 0.8061 |
| RP_POSITIVO                | 0.9275    | 0.9339 | 0.9307 |
| TIPO_HISTOPATOLOGICO       | 0.7557    | 0.8479 | 0.7992 |

## BCE + Grouped Softmax

Token level analysis
---

### Global Metrics

| Metric                 | Value     |
|------------------------|----------:|
| micro_precision        | 0.7781    |
| micro_recall           | 0.8552    |
| micro_f1               | 0.8148    |
| macro_precision        | 0.7138    |
| macro_recall           | 0.7737    |
| macro_f1               | 0.7396    |
| macro_pr_auc           | 0.7491    |
| micro_pr_auc           | 0.7862    |
| best_thresholds_mean   | 0.6142    |

### Per-Entity Metrics

| Entity                     | Precision | Recall | F1     |
|----------------------------|----------:|-------:|-------:|
| BRCA_NEGATIVO              | 0.5870    | 0.6027 | 0.5947 |
| BRCA_POSITIVO              | 0.5750    | 0.5542 | 0.5644 |
| CIRURGIA                   | 0.8155    | 0.8687 | 0.8412 |
| HER2_NEGATIVO              | 0.8750    | 0.7510 | 0.8082 |
| HER2_POSITIVO              | 0.6718    | 0.9592 | 0.7902 |
| POS_MENOPAUSA              | 0.5567    | 0.6736 | 0.6096 |
| PRE_MENOPAUSA              | 0.3361    | 0.3504 | 0.3431 |
| RE_NEGATIVO                | 0.7682    | 0.8818 | 0.8211 |
| RE_POSITIVO                | 0.9275    | 0.9333 | 0.9304 |
| RP_NEGATIVO                | 0.7814    | 0.8847 | 0.8299 |
| RP_POSITIVO                | 0.9334    | 0.9463 | 0.9398 |
| TIPO_HISTOPATOLOGICO       | 0.7375    | 0.8791 | 0.8021 |


Span level analysis
---

### Global Metrics

| Metric            | Value     |
|------------------|----------:|
| micro_precision  | 0.5679    |
| micro_recall     | 0.6942    |
| micro_f1         | 0.6247    |
| macro_precision  | 0.4833    |
| macro_recall     | 0.6461    |
| macro_f1         | 0.5406    |

### Per-Entity Metrics

| Entity                     | TP  | FP  | FN  | Precision | Recall | F1     |
|----------------------------|----:|----:|----:|----------:|-------:|-------:|
| BRCA_NEGATIVO              |   5 |  47 |  15 | 0.0962    | 0.2500 | 0.1389 |
| BRCA_POSITIVO              |   6 |  27 |   6 | 0.1818    | 0.5000 | 0.2667 |
| CIRURGIA                   | 289 | 202 | 185 | 0.5886    | 0.6097 | 0.5990 |
| HER2_NEGATIVO              | 146 |  45 |  46 | 0.7644    | 0.7604 | 0.7624 |
| HER2_POSITIVO              |  93 |  82 |  21 | 0.5314    | 0.8158 | 0.6436 |
| POS_MENOPAUSA              |  36 |  79 |  18 | 0.3130    | 0.6667 | 0.4260 |
| PRE_MENOPAUSA              |   5 |  67 |  15 | 0.0694    | 0.2500 | 0.1087 |
| RE_NEGATIVO                |  49 |  37 |  13 | 0.5698    | 0.7903 | 0.6622 |
| RE_POSITIVO                | 203 |  51 |  26 | 0.7992    | 0.8865 | 0.8406 |
| RP_NEGATIVO                |  76 |  47 |  22 | 0.6179    | 0.7755 | 0.6878 |
| RP_POSITIVO                | 163 |  39 |  18 | 0.8069    | 0.9006 | 0.8512 |
| TIPO_HISTOPATOLOGICO       | 225 | 263 | 186 | 0.4611    | 0.5474 | 0.5006 |
